# Generalized Linear Models (GLMs)

## Learning Objectives
- Understand the three components of a GLM: **random component** (exponential family distribution), **systematic component** (linear predictor $\eta = \theta^T x$), and **link function** $g(\mu) = \eta$
- Recognise linear regression, logistic regression, and Poisson regression as special cases sharing a **universal gradient form** $\nabla\ell = X^T(y - \mu)$
- Derive the **IRLS update rule** $(X^T W X)^{-1} X^T(y - \mu)$ as Newton-Raphson applied to the GLM log-likelihood
- Understand the role of the **variance function** $V(\mu)$ and **link derivative** $g'(\mu)$ in the IRLS weight $w_i = 1 / [V(\mu_i) g'(\mu_i)^2]$
- Implement `fit_glm` in NumPy using IRLS for Gaussian, Binomial, and Poisson families

## Problem Statement

Given a training set $\{(x^{(i)}, y^{(i)})\}_{i=1}^{n}$ where $y^{(i)}$ follows an **exponential family distribution**, find parameters $\theta$ that maximise the log-likelihood:

$\displaystyle \theta^* = \arg\max_{\theta}\; \ell(\theta) = \sum_{i=1}^{n} \log p(y^{(i)} \mid x^{(i)}; \theta)$

---

### Why GLMs?

Linear regression assumes a Gaussian response — unsuitable when:
- $y \in \{0,1\}$ (binary outcomes → Bernoulli)
- $y \in \{0, 1, 2, \ldots\}$ (count outcomes → Poisson)
- $y > 0$ (positive continuous → Gamma)

GLMs generalise linear regression by replacing the Gaussian assumption with any **exponential family** distribution, keeping the linear predictor $\eta = \theta^T x$.

---

### Core Idea

A GLM is defined by three choices:

| Component | Definition | Controls |
|---|---|---|
| **Random** | Distribution of $y$ (exponential family) | Variance structure |
| **Systematic** | Linear predictor $\eta = \theta^T x$ | Signal |
| **Link** | $g(\mu) = \eta$ mapping mean to predictor | Non-linearity |

### Canonical GLM Families

| Model | Distribution | Canonical link $g$ | Mean $\mu = g^{-1}(\eta)$ | Variance $V(\mu)$ |
|---|---|---|---|---|
| Linear Regression | Gaussian $\mathcal{N}(\mu, \sigma^2)$ | Identity: $\eta$ | $\mu = \eta$ | $1$ |
| Logistic Regression | Bernoulli $\text{Bern}(\mu)$ | Logit: $\log\tfrac{\mu}{1-\mu}$ | $\mu = \sigma(\eta)$ | $\mu(1-\mu)$ |
| Poisson Regression | Poisson$(\mu)$ | Log: $\log \mu$ | $\mu = e^\eta$ | $\mu$ |

**Universal gradient**: Across all canonical GLMs, the gradient of $\ell$ takes the same form:

$\displaystyle \nabla_\theta \ell = X^T(y - \mu)$

The only difference between models is how $\mu$ is computed from $\theta$.

In [ ]:
from IPython.display import SVG, display

svg = """
<svg xmlns="http://www.w3.org/2000/svg" width="640" height="310" viewBox="0 0 640 310">
  <rect width="640" height="310" fill="#fafafa" rx="8"/>
  <text x="320" y="24" text-anchor="middle" font-size="13" font-weight="bold" fill="#222">GLM Framework &#x2014; Three Components</text>

  <!-- Box 1: Random Component -->
  <rect x="20" y="50" width="170" height="190" rx="6" fill="#e8f4fd" stroke="#4a90d9" stroke-width="2"/>
  <text x="105" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#4a90d9">Random Component</text>
  <line x1="30" y1="80" x2="180" y2="80" stroke="#4a90d9" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="105" y="100" text-anchor="middle" font-size="10" fill="#333">y ~ Exponential Family</text>
  <text x="105" y="125" text-anchor="middle" font-size="9" fill="#555">Gaussian: y ~ N(&#x3BC;,&#x3C3;&#xB2;)</text>
  <text x="105" y="145" text-anchor="middle" font-size="9" fill="#555">Bernoulli: y ~ Bern(&#x3BC;)</text>
  <text x="105" y="165" text-anchor="middle" font-size="9" fill="#555">Poisson: y ~ Pois(&#x3BC;)</text>
  <text x="105" y="195" text-anchor="middle" font-size="9" fill="#777">Variance: V(&#x3BC;)</text>
  <text x="105" y="213" text-anchor="middle" font-size="9" fill="#777">Gaussian: V=1</text>
  <text x="105" y="228" text-anchor="middle" font-size="9" fill="#777">Bernoulli: V=&#x3BC;(1-&#x3BC;)</text>

  <!-- Arrow 1 -->
  <line x1="192" y1="145" x2="228" y2="145" stroke="#888" stroke-width="2"/>
  <polygon points="228,140 238,145 228,150" fill="#888"/>
  <text x="210" y="138" text-anchor="middle" font-size="8" fill="#888">link g</text>

  <!-- Box 2: Link Function -->
  <rect x="240" y="50" width="160" height="190" rx="6" fill="#fff8e8" stroke="#e07b00" stroke-width="2"/>
  <text x="320" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#e07b00">Link Function</text>
  <line x1="250" y1="80" x2="390" y2="80" stroke="#e07b00" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="320" y="100" text-anchor="middle" font-size="10" fill="#333">g(&#x3BC;) = &#x3B7; = &#x3B8;&#x1D40;x</text>
  <text x="320" y="130" text-anchor="middle" font-size="9" fill="#555">Identity: &#x3B7; = &#x3BC;</text>
  <text x="320" y="152" text-anchor="middle" font-size="9" fill="#555">Logit: &#x3B7; = log[&#x3BC;/(1-&#x3BC;)]</text>
  <text x="320" y="174" text-anchor="middle" font-size="9" fill="#555">Log: &#x3B7; = log(&#x3BC;)</text>
  <line x1="250" y1="192" x2="390" y2="192" stroke="#e07b00" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="320" y="211" text-anchor="middle" font-size="9" fill="#777">Inverse: &#x3BC; = g&#x207B;&#xB9;(&#x3B7;)</text>
  <text x="320" y="228" text-anchor="middle" font-size="9" fill="#777">Gaussian: &#x3BC;=&#x3B7;  Bern: &#x3BC;=&#x3C3;(&#x3B7;)  Pois: &#x3BC;=e&#x207F;</text>

  <!-- Arrow 2 -->
  <line x1="402" y1="145" x2="438" y2="145" stroke="#888" stroke-width="2"/>
  <polygon points="438,140 448,145 438,150" fill="#888"/>
  <text x="420" y="138" text-anchor="middle" font-size="8" fill="#888">&#x3B7;=&#x3B8;&#x1D40;x</text>

  <!-- Box 3: Systematic Component -->
  <rect x="450" y="50" width="170" height="190" rx="6" fill="#edfaed" stroke="#1a6e2e" stroke-width="2"/>
  <text x="535" y="74" text-anchor="middle" font-size="11" font-weight="bold" fill="#1a6e2e">Systematic Component</text>
  <line x1="460" y1="80" x2="610" y2="80" stroke="#1a6e2e" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="535" y="105" text-anchor="middle" font-size="10" fill="#333">&#x3B7; = &#x3B8;&#x1D40;x = &#x3B8;&#x2080; + &#x3B8;&#x2081;x&#x2081; + &#x2026;</text>
  <text x="535" y="135" text-anchor="middle" font-size="9" fill="#555">Parameters &#x3B8; &#x2208; &#x211D;&#x1D48;&#x207A;&#xB9;</text>
  <text x="535" y="160" text-anchor="middle" font-size="9" fill="#555">Same linear form</text>
  <text x="535" y="178" text-anchor="middle" font-size="9" fill="#555">for ALL GLM families</text>
  <line x1="460" y1="195" x2="610" y2="195" stroke="#1a6e2e" stroke-width="1" stroke-dasharray="3,2"/>
  <text x="535" y="215" text-anchor="middle" font-size="9" fill="#777">Universal gradient:</text>
  <text x="535" y="232" text-anchor="middle" font-size="9" font-weight="bold" fill="#1a6e2e">&#x2207;&#x2113; = X&#x1D40;(y &#x2212; &#x3BC;)</text>

  <!-- Bottom label -->
  <text x="320" y="275" text-anchor="middle" font-size="10" fill="#333">IRLS training: &#x3B8; &#x2190; &#x3B8; + (X&#x1D40;WX)&#x207B;&#xB9; X&#x1D40;(y &#x2212; &#x3BC;)&#x2002;&#x2002;W = diag(w&#x1D62;),&#x2002; w&#x1D62; = 1 / [V(&#x3BC;&#x1D62;) g&#x2019;(&#x3BC;&#x1D62;)&#xB2;]</text>
</svg>
"""

display(SVG(svg))

## Model Specification

For a query point $x \in \mathbb{R}^{d+1}$ (with $x_0 = 1$ prepended), a GLM predicts:

$\displaystyle \hat{y} = \mu = g^{-1}(\theta^T x)$

where $g$ is the link function for the chosen family. The **exponential family** log-likelihood takes the form:

$\displaystyle \log p(y \mid \theta, \phi) = \frac{y\,b(\theta) - a(\theta)}{\phi} + c(y, \phi)$

**Key property**: With the canonical link $g = b'^{-1}$, the natural parameter equals the linear predictor $\theta^T x$, yielding the universal gradient $\nabla\ell = X^T(y - \mu)$.

| Quantity | Gaussian | Bernoulli | Poisson |
|---|---|---|---|
| Natural parameter $\eta$ | $\mu$ | $\log\tfrac{\mu}{1-\mu}$ | $\log\mu$ |
| Mean $\mu$ | $\eta$ | $\sigma(\eta)$ | $e^\eta$ |
| Variance $V(\mu)$ | $1$ | $\mu(1-\mu)$ | $\mu$ |
| Link derivative $g'(\mu)$ | $1$ | $\tfrac{1}{\mu(1-\mu)}$ | $\tfrac{1}{\mu}$ |
| IRLS weight $w_i$ | $1$ | $\mu_i(1-\mu_i)$ | $\mu_i$ |

**Note**: The IRLS weight $w_i = 1 / [V(\mu_i) g'(\mu_i)^2]$ simplifies to $V(\mu_i)$ for canonical links since $g'(\mu_i) = 1/V(\mu_i)$.

## Derivation

**High-level steps:**
1. Write the log-likelihood for an exponential family response
2. Differentiate with respect to $\theta$ using the chain rule
3. Show the gradient collapses to $X^T(y - \mu)$ for canonical links
4. Apply Newton-Raphson to obtain the IRLS update

---

**Step 1 — Log-likelihood**

$\displaystyle \ell(\theta) = \sum_{i=1}^{n} \log p(y^{(i)} \mid \mu_i), \quad \mu_i = g^{-1}(\theta^T x^{(i)})$

---

**Step 2 — Gradient via chain rule**

$\displaystyle \frac{\partial \ell}{\partial \theta_j} = \sum_{i=1}^{n} \frac{\partial \log p}{\partial \mu_i} \cdot \frac{\partial \mu_i}{\partial \eta_i} \cdot \frac{\partial \eta_i}{\partial \theta_j} = \sum_{i=1}^{n} \frac{y_i - \mu_i}{V(\mu_i)} \cdot \frac{1}{g'(\mu_i)} \cdot x_j^{(i)}$

---

**Step 3 — Canonical link simplification**

For canonical links $g'(\mu) = 1/V(\mu)$, so the factor $V(\mu_i) g'(\mu_i) = 1$ and the sum collapses:

$\displaystyle \boxed{\nabla_\theta \ell = X^T(y - \mu)}$

This is the **universal gradient** — identical in form for linear, logistic, and Poisson regression.

---

**Step 4 — Hessian and Newton step (IRLS)**

Differentiating the gradient again:

$\displaystyle H = \nabla^2_\theta \ell = -X^T W X, \quad W = \mathrm{diag}(w_i), \quad w_i = \frac{1}{V(\mu_i)\, g'(\mu_i)^2}$

Newton step $\theta \leftarrow \theta - H^{-1}\nabla\ell$:

$\displaystyle \boxed{\theta^{(t+1)} = \theta^{(t)} + (X^T W X)^{-1} X^T(y - \mu)}$

This is the **IRLS update** — Newton-Raphson specialised to GLMs. Each iteration re-computes $\mu$ and $W$ using the current $\theta$, then solves a weighted least squares problem.

## Algorithm: Iteratively Reweighted Least Squares (IRLS)

IRLS is Newton-Raphson applied to the GLM log-likelihood. It converges quadratically and requires no learning rate.

**Step 1 — Initialise**

Set $\theta \leftarrow \mathbf{0} \in \mathbb{R}^{d+1}$. Prepend a bias column of ones to $X$.

**Step 2 — Compute mean response**

$\displaystyle \eta = X\theta \in \mathbb{R}^n, \qquad \mu = g^{-1}(\eta) \in \mathbb{R}^n$

**Step 3 — Compute IRLS weights**

$\displaystyle w_i = \frac{1}{V(\mu_i)\, g'(\mu_i)^2}$

For canonical links: $w_i = V(\mu_i)$ (e.g., $1$ for Gaussian, $\mu_i(1-\mu_i)$ for Bernoulli, $\mu_i$ for Poisson).

**Step 4 — Newton step (solve weighted least squares)**

$\displaystyle \Delta\theta = (X^T W X)^{-1} X^T(y - \mu), \qquad \theta \leftarrow \theta + \Delta\theta$

Use `np.linalg.solve(X.T @ W @ X, X.T @ (y - mu))` for numerical stability.

**Step 5 — Repeat** steps 2–4 until $\|\nabla\ell\| = \|X^T(y-\mu)\| < \varepsilon$ or maximum iterations reached.

**Convergence note**: Gaussian GLM converges in **1 iteration** (loss is quadratic); Bernoulli and Poisson typically need 5–15 iterations.

| Quantity | Formula | Shape |
|----------|---------|-------|
| Linear predictor | $\eta = X\theta$ | $(n)$ |
| Mean response | $\mu = g^{-1}(\eta)$ | $(n)$ |
| Weights | $w_i = V(\mu_i)$ (canonical) | $(n)$ |
| Gradient | $g = X^T(y - \mu)$ | $(d+1)$ |
| Hessian | $H = -X^T \mathrm{diag}(w) X$ | $(d+1, d+1)$ |
| IRLS step | $\Delta\theta = -H^{-1}g$ | $(d+1)$ |

## Key Properties

**Universal gradient** — for canonical links, $\nabla\ell = X^T(y - \mu)$ regardless of the response family. Only the computation of $\mu$ from $\theta$ changes between models.

**IRLS = Newton-Raphson** — the IRLS update is Newton's method applied to the GLM log-likelihood. It inherits **quadratic convergence** near the optimum (5–15 iterations vs hundreds for gradient ascent).

**Gaussian special case** — the MSE Hessian $H = -X^TX$ is constant (no $W$ dependence on $\mu$). IRLS converges in **exactly one step** from any $\theta_0$, yielding the Normal Equation $\theta^* = (X^TX)^{-1}X^Ty$.

**Log-likelihood is concave** — for canonical link GLMs with exponential family responses, the log-likelihood is globally concave in $\theta$, guaranteeing IRLS converges to the unique global maximum.

**Deviance** — the goodness-of-fit statistic for GLMs is the **deviance** $D = 2[\ell(\text{saturated}) - \ell(\hat{\theta})]$, which generalises the residual sum of squares from linear regression.

**Gradient ascent vs IRLS** — both maximise the same log-likelihood. Gradient ascent uses only $\nabla\ell$; IRLS also uses $H = -X^TWX$. IRLS converges far faster but costs $O(nd^2 + d^3)$ per iteration vs $O(nd)$ for gradient ascent.

In [ ]:
import numpy as np


# ── Family specifications ────────────────────────────────────────────────────

def _gaussian(eta):
    mu = eta
    w  = np.ones_like(mu)           # V(mu) = 1
    return mu, w

def _binomial(eta):
    mu = 1.0 / (1.0 + np.exp(-np.clip(eta, -500, 500)))
    w  = mu * (1.0 - mu)            # V(mu) = mu(1-mu)
    return mu, w

def _poisson(eta):
    mu = np.exp(np.clip(eta, -500, 20))
    w  = mu                         # V(mu) = mu
    return mu, w

_FAMILIES = {
    'gaussian': _gaussian,
    'binomial': _binomial,
    'poisson':  _poisson,
}


def fit_glm(X, y, family='binomial', max_iter=25, tol=1e-8):
    """
    Fit a Generalized Linear Model via IRLS (Newton-Raphson on the log-likelihood).
    Assumes canonical link for the chosen family.

    Inputs
    ------
    X        : np.ndarray, shape (n, d+1)  — design matrix with bias column (x_0=1) prepended
    y        : np.ndarray, shape (n,)      — response vector
                 gaussian  : continuous real
                 binomial  : binary {0, 1}
                 poisson   : non-negative integer counts
    family   : str  — one of 'gaussian', 'binomial', 'poisson'
    max_iter : int  — maximum IRLS iterations (Gaussian converges in 1)
    tol      : float — stop when ||X^T(y - mu)|| < tol

    Outputs
    -------
    theta   : np.ndarray, shape (d+1,)   — learned parameters (bias first)
    history : list of (int, float)       — (iteration, log-likelihood) at each step
    """
    if family not in _FAMILIES:
        raise ValueError(f"family must be one of {list(_FAMILIES)}")

    family_fn = _FAMILIES[family]
    n, d = X.shape
    theta = np.zeros(d)
    history = []

    for t in range(max_iter):
        eta      = X @ theta                           # (n,) linear predictor
        mu, w    = family_fn(eta)                      # (n,) mean and IRLS weights
        grad     = X.T @ (y - mu)                      # (d+1,) universal gradient ∇ℓ = Xᵀ(y−μ)
        H        = -(X.T * w) @ X                      # (d+1,d+1) Hessian H = −XᵀWX
        delta    = np.linalg.solve(-H, grad)            # (d+1,) Newton step
        theta    = theta + delta

        history.append((t, float(np.sum(y * eta - _log_partition(eta, family)))))

        if np.linalg.norm(grad) < tol:
            break

    return theta, history


def _log_partition(eta, family):
    """Log-partition term a(θ) for log-likelihood = y·θ - a(θ) + c(y)."""
    if family == 'gaussian':
        return 0.5 * eta ** 2
    elif family == 'binomial':
        return np.log1p(np.exp(np.clip(eta, -500, 500)))
    elif family == 'poisson':
        return np.exp(np.clip(eta, -500, 20))


def predict(X, theta, family='binomial', threshold=0.5):
    """
    Return predictions on the mean scale (μ = g⁻¹(Xθ)).
    For binomial, also applies decision threshold to return {0, 1}.

    Inputs
    ------
    X         : np.ndarray, shape (n, d+1)  — design matrix with bias column (x_0=1) prepended
    theta     : np.ndarray, shape (d+1,)    — model parameters
    family    : str                         — 'gaussian', 'binomial', or 'poisson'
    threshold : float                       — decision threshold for binomial (default 0.5)

    Output
    ------
    mu    : np.ndarray, shape (n,)  — predicted means (probabilities for binomial)
    y_hat : np.ndarray, shape (n,)  — class labels for binomial, same as mu otherwise
    """
    eta     = X @ theta
    mu, _   = _FAMILIES[family](eta)
    if family == 'binomial':
        return mu, (mu >= threshold).astype(int)
    return mu, mu